# SVD Recommendation – REES46
Phương pháp: **Truncated SVD** (scipy) – đơn giản, nhanh, phù hợp data lớn  
Metric: **Recall@20** và **Recall@50** trên val & test set

In [2]:
!git clone https://huggingface.co/datasets/nguyenmaiductrong/rees46-processed-data

Cloning into 'rees46-processed-data'...
remote: Enumerating objects: 25, done.
remote: Total 25 (delta 0), reused 0 (delta 0), pack-reused 25 (from 1)
Receiving objects: 100% (25/25), 1.77 MiB | 5.54 MiB/s, done.
Filtering content: 100% (13/13), 675.87 MiB | 114.39 MiB/s, done.


In [3]:
import numpy as np
import json, pickle, os, time
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.linalg import svds

data_dir = "/content/rees46-processed-data/temporal"

## 1. Load dữ liệu

In [4]:
view_src     = np.load(f"{data_dir}/view_train_src.npy")
view_dst     = np.load(f"{data_dir}/view_train_dst.npy")
cart_src     = np.load(f"{data_dir}/cart_train_src.npy")
cart_dst     = np.load(f"{data_dir}/cart_train_dst.npy")
purchase_src = np.load(f"{data_dir}/purchase_train_src.npy")
purchase_dst = np.load(f"{data_dir}/purchase_train_dst.npy")

val_user     = np.load(f"{data_dir}/val_user_idx.npy")
val_product  = np.load(f"{data_dir}/val_product_idx.npy")
test_user    = np.load(f"{data_dir}/test_user_idx.npy")
test_product = np.load(f"{data_dir}/test_product_idx.npy")

with open(f"{data_dir}/node_counts.json") as f:
    node_counts = json.load(f)

num_users = node_counts['user']
num_items = node_counts['product']
print(f"Users: {num_users:,} | Items: {num_items:,}")
print(f"Val: {len(val_user):,} | Test: {len(test_user):,}")

Users: 267,973 | Items: 60,798
Val: 415,195 | Test: 188,112


## 2. Xây dựng ma trận tương tác

Trọng số: view=1, cart=2, purchase=3

In [5]:
src = np.concatenate([view_src, cart_src, purchase_src])
dst = np.concatenate([view_dst, cart_dst, purchase_dst])
w   = np.concatenate([
    np.ones(len(view_src)),
    np.ones(len(cart_src))     * 2,
    np.ones(len(purchase_src)) * 3
])

R = csr_matrix(
    coo_matrix((w, (src, dst)), shape=(num_users, num_items))
)
print(f"R shape: {R.shape} | nnz: {R.nnz:,} | density: {R.nnz/(num_users*num_items)*100:.4f}%")

R shape: (267973, 60798) | nnz: 15,045,001 | density: 0.0923%


## 3. SVD – Giảm chiều

`R ≈ U · diag(S) · Vt`  
→ `user_emb = U · √S`  
→ `item_emb = Vt.T · √S`

In [6]:
K = 64  # số chiều embedding – có thể thử 32, 64, 128

print(f"Đang chạy SVD với K={K}...")
t0 = time.time()

U, S, Vt = svds(R, k=K)

# svds trả về theo thứ tự tăng dần → đảo lại
order = np.argsort(-S)
U, S, Vt = U[:, order], S[order], Vt[order, :]

sqrt_S = np.sqrt(S)
user_emb = U   @ np.diag(sqrt_S)   # (num_users, K)
item_emb = Vt.T @ np.diag(sqrt_S)  # (num_items, K)

print(f"Xong! Thời gian: {time.time()-t0:.1f}s")
print(f"user_emb: {user_emb.shape} | item_emb: {item_emb.shape}")
print(f"Top singular values: {S[:5].round(1)}")

Đang chạy SVD với K=64...
Xong! Thời gian: 5.7s
user_emb: (267973, 64) | item_emb: (60798, 64)
Top singular values: [9614.7 7627.  6140.5 5534.6 5321.8]


## 4. Recall@K – Đánh giá

In [7]:
def recall_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=1024):
    hits = 0
    n = len(user_idx)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores   = user_emb[u_batch] @ item_emb.T
        top_k    = np.argpartition(-scores, k, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            if gt in top_k[i]:
                hits += 1
    return hits / n


def ndcg_at_k(user_emb, item_emb, user_idx, true_items, k=20, batch_size=1024):
    """
    NDCG@K – mỗi user chỉ có 1 ground-truth item.
    NDCG@K = 1 / log2(rank + 1)  nếu item nằm trong top-K, ngược lại = 0.
    """
    total = 0.0
    n = len(user_idx)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch  = user_idx[start:end]
        gt_batch = true_items[start:end]
        scores    = user_emb[u_batch] @ item_emb.T
        top_k_idx = np.argsort(-scores, axis=1)[:, :k]
        for i, gt in enumerate(gt_batch):
            ranks = np.where(top_k_idx[i] == gt)[0]
            if len(ranks) > 0:
                total += 1.0 / np.log2(ranks[0] + 2)
    return total / n


def evaluate_all(user_emb, item_emb, user_idx, true_items, label="", batch_size=1024):
    """In đầy đủ Recall@10/20/50 và NDCG@10/20/50."""
    r10 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    r20 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    r50 = recall_at_k(user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    n10 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=10,  batch_size=batch_size)
    n20 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=20,  batch_size=batch_size)
    n50 = ndcg_at_k  (user_emb, item_emb, user_idx, true_items, k=50,  batch_size=batch_size)
    hdr = f"  {label}  " if label else "  "
    print(f"\n{'='*52}")
    print(f"{hdr}Recall@10  : {r10:.4f}")
    print(f"{hdr}Recall@20  : {r20:.4f}")
    print(f"{hdr}Recall@50  : {r50:.4f}")
    print(f"{hdr}NDCG@10    : {n10:.4f}")
    print(f"{hdr}NDCG@20    : {n20:.4f}")
    print(f"{hdr}NDCG@50    : {n50:.4f}")
    print(f"{'='*52}")
    return dict(r10=r10, r20=r20, r50=r50, n10=n10, n20=n20, n50=n50)


In [8]:
print("Đang đánh giá trên Val set...")
t0 = time.time()
val_metrics = evaluate_all(user_emb, item_emb, val_user, val_product, label="VAL")
print(f"  Thời gian       : {time.time()-t0:.1f}s")

Đang đánh giá trên Val set...

  VAL  Recall@10  : 0.2214
  VAL  Recall@20  : 0.2634
  VAL  Recall@50  : 0.3202
  VAL  NDCG@10    : 0.1445
  VAL  NDCG@20    : 0.1551
  VAL  NDCG@50    : 0.1664
  Thời gian       : 1312.3s


In [9]:
print("Đang đánh giá trên Test set...")
t0 = time.time()
test_metrics = evaluate_all(user_emb, item_emb, test_user, test_product, label="TEST")
print(f"  Thời gian       : {time.time()-t0:.1f}s")


Đang đánh giá trên Test set...

  TEST  Recall@10  : 0.1075
  TEST  Recall@20  : 0.1376
  TEST  Recall@50  : 0.1837
  TEST  NDCG@10    : 0.0634
  TEST  NDCG@20    : 0.0710
  TEST  NDCG@50    : 0.0801
  Thời gian       : 594.2s
